# Milestone 4: forecast and evaluation

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It follows course modules 9 and 10: forecast the test period, then score the forecast. It adds what the course leaves out: a walk-forward one-step forecast beside the course's static one, the out-of-sample window 2024-01-02 to 2026-09-30, a random walk under every metric, a Diebold-Mariano test against it, and the coverage of the intervals.

It reads committed files in `app/public/data/`: `evaluation.json` and the forecast files `forecast_<model>_<method>.json`. One cell near the end rebuilds them from `prices.json`, `fit.json` and the residual files (all committed) and checks they match; it skips if `prices.json` is absent.

The pictures are in the app (`cd app && npm run dev`): the **Forecast and evaluation** page.

In [ ]:
from arima_forecasting.artifacts import find_app_data, read_artifact

app_data = find_app_data()
ev = read_artifact(app_data, "evaluation")["data"]
for w in ev["windows"]:
    print(f"{w['id']:>4}: {w['start']} to {w['end']}, {w['n']} days, forecast from {w['origin_date']}")
for m in ev["models"]:
    print(f"{m['id']:>6}: {m['label']}; methods {m['methods']}; residual t fit df {m['t_df']:.2f}")

## Module 9: the static forecast, and why the course's code stops

The course calls `model_fit.get_forecast(steps=len(test_data))` on a model fitted to a pandas object whose index is the trading dates. Trading dates have no fixed frequency (holidays), and statsmodels 0.15 then refuses to forecast past the sample: "No supported index is available". Here every model is built on a plain array and the dates are put back afterwards (`forecasting.py`).

The static path is one forecast from one origin: for the test window the last training day; for the out-of-sample window the last test day, with the state filtered through the test rows at the training parameters.

In [ ]:
aic_static = read_artifact(app_data, "forecast_aic_static")["data"]
for w in aic_static["windows"]:
    first, last = w["rows"][0], w["rows"][-1]
    print(f"{w['id']}: origin {w['origin']['date']} at ${w['origin']['close_usd']:.2f}; day 1 median ${first['median_usd']:.2f} (sd {first['sd_log']:.4f}); day {w['n']} median ${last['median_usd']:.2f}, 95% ${last['lower_95_usd']:.2f} to ${last['upper_95_usd']:.2f}, actual ${last['actual_usd']:.2f}")

## The median and the mean of a dollar forecast

The models forecast the log price: a normal with mean mu and variance s^2. The price is then log-normal, and exp(mu), what the course reports, is its **median**. Its mean is exp(mu + s^2 / 2) (PR/FAQ, mistake 4). One day ahead s^2 is small and the two hardly differ; at the end of a long static path they do.

In [ ]:
for row in ev["metrics"]:
    if row["model_id"] == "aic":
        bt = row["back_transform"]
        print(f"aic {row['method']:>12} {row['window']:>4}: mean above median by up to {bt['max_pct']:.3f}% (on average {bt['mean_pct']:.3f}%)")

## Module 10: the metrics, with the random walk under each

The course computes RMSE, MAE and MAPE in dollars and prints them with nothing to compare. Its MAPE line prints `nan`: it subtracts a one-column DataFrame from a Series, and pandas lines the Series up against the column labels. Here each metric is computed on one-dimensional arrays, in dollars (on the median and on the mean) and in log points, and divided by the random walk's on the same days: below 1 the model is closer.

In [ ]:
rw = {(r["kind"], r["window"]): r for r in ev["random_walk"]}
kinds = {m["id"]: m["kind"] for m in ev["methods"]}
for row in ev["metrics"]:
    base = rw[(kinds[row["method"]], row["window"])]
    print(f"{row['model_id']:>6} {row['method']:>12} {row['window']:>4}: RMSE ${row['usd_median']['rmse']:7.3f} (random walk ${base['usd']['rmse']:7.3f}, ratio {row['ratio_to_rw']['usd_median']['rmse']:.4f}); RMSE log {row['log']['rmse']:.5f} (ratio {row['ratio_to_rw']['log']['rmse']:.4f}); MAPE {row['usd_median']['mape_pct']:.3f}%")

## Diebold-Mariano against the random walk

For each one-step forecast: the loss differential d_t = loss(model error) - loss(random-walk error), in log points, squared and absolute. DM = mean(d) / sqrt(gamma_0 / n), scaled by Harvey, Leybourne and Newbold's sqrt((n - 1) / n) and compared with a Student t on n - 1 degrees of freedom, two-sided. A rejection says the expected losses differ over this window; "no_difference" is not evidence that they are equal.

In [ ]:
for r in ev["dm"]:
    tested = "not tested" if r["p_value"] is None else f"DM* {r['statistic_hln']: .3f} on {r['df']} df, p {r['p_value']:.4f}"
    print(f"{r['model_id']:>6} {r['method']:>12} {r['window']:>4} {r['loss']:>8}: mean d {r['mean_diff']: .3e}; {tested}; {r['verdict']}")

## Interval coverage, normal and Student-t

How many actual log prices fall inside each interval, beside its nominal level. The t interval uses the degrees of freedom of the milestone-3 t fit and is scaled to the model's forecast variance. For one-step forecasts Kupiec's test asks whether the count is consistent with the level; a static path's days are not independent, so it is shown, not tested.

In [ ]:
for c in ev["coverage"]:
    if c["model_id"] == "aic":
        print(f"aic {c['method']:>12} {c['window']:>4} {c['distribution']:>6} {c['level']:.0%}: {c['inside']:>3} of {c['n']} inside ({c['rate']:.3f}); {c['verdict']}")

## Rebuild the files from the committed inputs

`forecast_payload` filters each model at its committed parameters over the committed rows; `evaluation_payload` scores the forecast files. The refit file takes about 12 seconds to rebuild (43 refits), so it is checked from a sampled block in `tests/test_committed_artifacts.py` instead.

In [ ]:
from arima_forecasting.artifacts import table_from_prices
from arima_forecasting.evaluation import evaluation_payload
from arima_forecasting.forecasting import FORECASTS, forecast_payload

if not (app_data / "prices.json").is_file():
    print("prices.json is not in this copy - skipped")
else:
    table = table_from_prices(read_artifact(app_data, "prices")["data"])
    fit = read_artifact(app_data, "fit")["data"]
    models = {m["id"]: m for m in fit["models"]}
    for model_id, method in FORECASTS:
        if method == "refit":
            continue
        t_df = read_artifact(app_data, f"residuals_{model_id}")["data"]["t_fit"]["df"]
        name = f"forecast_{model_id}_{method}"
        print(f"{name}.json matches a fresh computation:", forecast_payload(table, models[model_id], t_df, method) == read_artifact(app_data, name)["data"])
    forecasts = {f"forecast_{m}_{meth}": read_artifact(app_data, f"forecast_{m}_{meth}")["data"] for m, meth in FORECASTS}
    print("evaluation.json matches a fresh computation:", evaluation_payload(forecasts, fit) == ev)

## What comes next

Milestone 5 collects the course's four mistakes in one place, "where the sample goes wrong", and measures what each one costs.